# **Complementary EDA Notebook**


---
## 0. Setup Environment

In [14]:
# Do not modify this code
%pip install uts_36106_core==0.0.3

from uts_36106_core.folders import *
from uts_36106_core.ipyrenders import *
import os

at = AtFolder(
    course_code=36106,
    assignment="AT3",
)
at.mount_gdrive()
at.create_folders()
if at.is_colab:
  os.chdir(f'{at.folder_path}')

import warnings
warnings.simplefilter(action='ignore')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).

You can now save your data files in: /content/gdrive/MyDrive/36106/assignment/AT3/data


---
## Student Information

In [15]:
group_name = "Group 52"
student_name = "Andres Ethorimn Lobarb"
student_id = "26029165"

In [16]:
# Do not modify this code
print_tile(size="h1", key='group_name', value=group_name)

In [17]:
# Do not modify this code
print_tile(size="h1", key='student_name', value=student_name)

In [18]:
# Do not modify this code
print_tile(size="h1", key='student_id', value=student_id)

---
## 0. Python Packages

### 0.a Install Additional Packages

> If you are using additional packages, you need to install them here using the command: `! pip install <package_name>`

In [19]:
# No additional packages: pandas, numpy, scipy and altair are available on Colab.

### 0.b Import Packages

In [20]:
import altair as alt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import kruskal, norm, spearmanr

In [21]:
# Utils

def forward_change(balances, account_col, date_col, balance_col, horizon):
    """Balance change from each month-end to the month-end `horizon` months later."""
    ordered = balances.sort_values([account_col, date_col])
    future = ordered.groupby(account_col)[balance_col].shift(-horizon)
    out = ordered[[account_col, date_col, balance_col]].copy()
    out["future_change"] = future - ordered[balance_col]
    out["month"] = pd.to_datetime(out[date_col]).dt.to_period("M")
    return out


def column_profile(df):
    """One row per column: type, missing count and share, distinct values."""
    return pd.DataFrame({
        "type": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "missing_share": df.isna().mean().round(4),
        "distinct": df.nunique(),
    })


def spearman_summary(x, y, label):
    """Spearman correlation with a Fisher-z 95% interval and a size label."""
    mask = x.notna() & y.notna()
    rho, p_value = spearmanr(x[mask], y[mask])
    n = int(mask.sum())
    half_width = 1.96 / np.sqrt(n - 3)
    low, high = np.tanh(np.arctanh(rho) - half_width), np.tanh(np.arctanh(rho) + half_width)
    size = abs(rho)
    magnitude = "negligible" if size < 0.1 else "small" if size < 0.3 else "medium" if size < 0.5 else "large"
    return pd.Series({"feature": label, "n": n, "rho": round(rho, 4), "ci_low": round(low, 4),
                      "ci_high": round(high, 4), "p_value": p_value, "magnitude": magnitude})


def holm_adjust(p_values):
    """Holm step-down adjustment of a list of p-values (family-wise error control)."""
    order = np.argsort(p_values)
    m = len(p_values)
    adjusted = np.empty(m)
    running = 0.0
    for rank, index in enumerate(order):
        running = max(running, (m - rank) * p_values[index])
        adjusted[index] = min(running, 1.0)
    return adjusted


def kruskal_epsilon(df, value_col, group_col):
    """Kruskal-Wallis H test across groups with epsilon-squared effect size."""
    groups = [g[value_col].dropna().to_numpy() for _, g in df.groupby(group_col)]
    h_stat, p_value = kruskal(*groups)
    n, k = sum(len(g) for g in groups), len(groups)
    epsilon_sq = (h_stat - k + 1) / (n - k)
    return pd.Series({"H": round(h_stat, 2), "groups": k, "n": n, "p_value": p_value,
                      "epsilon_squared": round(epsilon_sq, 6)})


def group_mean(df, group_col, target):
    """Mean of `target` with a 95% interval per group."""
    table = df.groupby(group_col, observed=True)[target].agg(["mean", "std", "size"]).reset_index()
    table = table.rename(columns={"size": "n"})
    half = 1.96 * table["std"] / np.sqrt(table["n"])
    table["low"], table["high"] = table["mean"] - half, table["mean"] + half
    return table


def interval_chart(table, x_field, x_title, y_title, reference, reference_label):
    """Group means with 95% intervals against a grey, labelled reference line.

    The reference is the overall mean: groups on the line show no effect of the feature."""
    base = alt.Chart(table).encode(x=alt.X(f"{x_field}:O", title=x_title, sort=None))
    intervals = base.mark_rule(color="#898781").encode(
        y=alt.Y("low:Q", title=y_title, scale=alt.Scale(zero=False)), y2="high:Q")
    points = base.mark_point(filled=True, color="#2a78d6").encode(
        y="mean:Q", tooltip=[x_field, alt.Tooltip("mean:Q", format=".0f"), alt.Tooltip("low:Q", format=".0f"),
                             alt.Tooltip("high:Q", format=".0f"), "n"])
    ref = pd.DataFrame({"y": [reference], "label": [reference_label]})
    line = alt.Chart(ref).mark_rule(color="#898781", strokeDash=[4, 4]).encode(y="y:Q")
    text = alt.Chart(ref).mark_text(align="left", dx=4, dy=-7, color="#898781").encode(
        y="y:Q", x=alt.value(0), text="label:N")
    return (intervals + points + line + text).properties(width=520, height=260)


def qq_frame(values):
    """Sample quantiles against normal quantiles (all points up to 500 values, else 199 levels).

    Robust standardisation (median, IQR / 1.349) keeps a few extreme values from flattening the plot;
    for normal data it equals the usual standardisation."""
    values = pd.Series(values).dropna()
    n = len(values)
    levels = (np.arange(1, n + 1) - 0.5) / n if n <= 500 else np.linspace(0.005, 0.995, 199)
    scale = (values.quantile(0.75) - values.quantile(0.25)) / 1.349
    scale = scale if scale > 0 else values.std()  # IQR is 0 when most values are equal
    z = (values - values.median()) / scale
    return pd.DataFrame({"theoretical": norm.ppf(levels), "sample": np.quantile(z, levels)})


def distribution_charts(values, label, n_bins, tail):
    """Histogram (the `tail` share at each end not drawn) beside a normal QQ plot, with shape statistics."""
    values = pd.Series(values).dropna()
    low, high = values.quantile(tail), values.quantile(1 - tail)
    shown = values[(values >= low) & (values <= high)]
    counts, edges = np.histogram(shown, bins=n_bins)
    hist = alt.Chart(pd.DataFrame({"start": edges[:-1], "end": edges[1:], "count": counts})).mark_bar(
        color="#2a78d6").encode(x=alt.X("start:Q", bin="binned", title=label), x2="end:Q",
                                y=alt.Y("count:Q", title="Count")).properties(width=300, height=220)
    diagonal = pd.DataFrame({"theoretical": [-3, 3], "sample": [-3, 3]})
    qq = (alt.Chart(qq_frame(values)).mark_point(color="#2a78d6", size=14).encode(
            x=alt.X("theoretical:Q", title="Normal quantile"), y=alt.Y("sample:Q", title="Robust-standardised sample quantile"))
          + alt.Chart(diagonal).mark_line(color="#898781", strokeDash=[4, 4]).encode(x="theoretical:Q", y="sample:Q")
          ).properties(width=300, height=220)
    print(f"{label}: n = {len(values)}, skewness = {values.skew():.2f}, excess kurtosis = {values.kurt():.2f}, "
          f"histogram omits {len(values) - len(shown)} values outside the {tail:.1%} to {1 - tail:.1%} quantiles")
    return alt.hconcat(hist, qq)

In [22]:
# Constants
HORIZON = 3   # provisional working horizon (months ahead), as in SB-1 to SB-2
LOOKBACK = 3  # months of repayment history counted before each cut-off
# Customer fields loaded: financial profile for the analysis, protected attributes for the ethics
# section only. Names, contacts and addresses are never loaded.
CUSTOMER_COLUMNS = ["customer_id", "income_band", "credit_score_band", "hardship_program",
                    "age", "gender", "vulnerable_customer_flag"]

---
## B. Data Understanding

**Approach.** The assigned tables gave us no signal for how a single balance moves next, so here we put the same question to three other tables: the customer's financial profile (B.2), their loans (B.3) and their recent repayment behaviour (B.4). Each finding comes from the outputs above it; decisions are taken after the EDA notebooks.

In [23]:
try:
  customers = pd.read_csv(at.folder_path / "customers.csv", usecols=CUSTOMER_COLUMNS)
  accounts = pd.read_csv(at.folder_path / "accounts.csv", usecols=["account_id", "customer_id"])
  loans = pd.read_csv(at.folder_path / "loans.csv",
                      usecols=["account_id", "loan_type", "loan_start_date", "loan_risk_grade", "debt_service_band"])
  repayments = pd.read_csv(at.folder_path / "loan_repayments.csv",
                           usecols=["account_id", "payment_date", "late_payment", "missed_payment"])
  balances = pd.read_csv(at.folder_path / "account_balance_history.csv",
                         usecols=["account_id", "balance_date", "balance"])
except Exception as e:
  print(e)

### B.1 Explore Dataset

In [24]:
all_customer_columns = pd.read_csv(at.folder_path / "customers.csv", nrows=0).columns
print("customers.csv columns:", len(all_customer_columns), "| loaded:", len(CUSTOMER_COLUMNS))
for name, table in [("customers", customers), ("loans", loans), ("loan_repayments", repayments)]:
    print(f"\n{name}: {table.shape[0]} rows")
    display(column_profile(table))

loans["loan_start_date"] = pd.to_datetime(loans["loan_start_date"])
repayments["payment_date"] = pd.to_datetime(repayments["payment_date"])
history_end = pd.to_datetime(balances["balance_date"]).max()
print("Accounts with a loan:", loans["account_id"].nunique(), "of", accounts["account_id"].nunique())
print("Loans starting after the balance history ends:", f"{(loans['loan_start_date'] > history_end).mean():.1%}")
in_window = repayments["payment_date"].between(pd.Timestamp("2022-01-01"), history_end)
print("Repayments dated", repayments["payment_date"].min().date(), "to", repayments["payment_date"].max().date(),
      f"| inside the balance history: {in_window.mean():.1%}")

customers.csv columns: 33 | loaded: 7

customers: 15000 rows


,type,missing,missing_share,distinct
customer_id,int64,0,0.0,15000
age,int64,0,0.0,67
gender,object,0,0.0,2
income_band,object,0,0.0,4
credit_score_band,object,0,0.0,5
vulnerable_customer_flag,int64,0,0.0,2
hardship_program,int64,0,0.0,2



loans: 10673 rows


,type,missing,missing_share,distinct
account_id,int64,0,0.0,9447
loan_type,object,0,0.0,5
loan_start_date,object,0,0.0,6834
loan_risk_grade,object,0,0.0,5
debt_service_band,object,0,0.0,4



loan_repayments: 1009766 rows


,type,missing,missing_share,distinct
account_id,int64,0,0.0,9447
payment_date,object,0,0.0,777
late_payment,int64,0,0.0,2
missed_payment,int64,0,0.0,2


Accounts with a loan: 9447 of 38562
Loans starting after the balance history ends: 17.0%
Repayments dated 2000-02-01 to 2064-10-01 | inside the balance history: 14.6%


In [25]:
start_years = loans["loan_start_date"].dt.year.value_counts().sort_index().rename_axis("year").reset_index(name="loans")
alt.Chart(start_years).mark_bar(color="#2a78d6").encode(
    x=alt.X("year:O", title="Loan start year"), y=alt.Y("loans:Q", title="Loans"), tooltip=["year", "loans"]
).properties(width=620, height=220)

alt.Chart(...)

In [26]:
dataset_insights = """
We bring in three more tables. From <customers> (15000 rows) we load only 7 of its 33 columns: the
financial profile, plus protected attributes for the ethics check; names, contacts and addresses
are never loaded. <loans> holds 10673 loans on 9447 of the 38562 accounts, and <loan_repayments>
holds 1009766 payments.

Their dates run well beyond the balance history: 17.0% of loans start after it ends, and only
14.6% of repayments fall inside it (2000 to 2064 overall; chart). We therefore count loans and
repayments only up to each cut-off.
"""

In [27]:
# Do not modify this code
print_tile(size="h3", key='dataset_insights', value=dataset_insights)

### B.2 Explore Feature of Interest `income_band`, `credit_score_band`, `hardship_program`

In [28]:
# One example per account and month-end with an observed change over the next HORIZON months
changes = forward_change(balances, "account_id", "balance_date", "balance", HORIZON).dropna(subset=["future_change"])
panel = changes.merge(accounts, on="account_id").merge(customers, on="customer_id", how="left")
print("Examples:", len(panel), "| accounts:", panel["account_id"].nunique())

profile = ["income_band", "credit_score_band", "hardship_program"]
counts = pd.concat([panel[c].astype(str).value_counts().rename_axis("level").reset_index(name="examples").assign(field=c)
                    for c in profile])
bars = alt.Chart(counts).mark_bar(color="#2a78d6").encode(
    x=alt.X("examples:Q", title="Examples"), y=alt.Y("level:N", title=None),
    row=alt.Row("field:N", title=None)).properties(width=300, height=90)
distribution_charts(panel["future_change"], f"Balance change over {HORIZON} months ($)", 40, 0.005) & bars

Examples: 1735290 | accounts: 38562
Balance change over 3 months ($): n = 1735290, skewness = 0.16, excess kurtosis = 3.38, histogram omits 17354 values outside the 0.5% to 99.5% quantiles


alt.VConcatChart(...)

In [29]:
profile_tests = pd.DataFrame({c: kruskal_epsilon(panel, "future_change", c) for c in profile}).T
profile_tests["p_holm"] = holm_adjust(profile_tests["p_value"].to_numpy())
display(profile_tests)

,H,groups,n,p_value,epsilon_squared,p_holm
income_band,4387.52,4.0,1735290.0,0.000000,0.002527,0.000000
credit_score_band,6.06,5.0,1735290.0,0.194862,0.000001,0.389723
hardship_program,1.06,2.0,1735290.0,0.302517,0.000000,0.389723


In [30]:
overall = panel["future_change"].mean()
order = {"income_band": ["Low", "Middle", "UpperMiddle", "Affluent"],
         "credit_score_band": ["Poor", "Fair", "Good", "Very Good", "Excellent"], "hardship_program": ["0", "1"]}
alt.hconcat(*[interval_chart(group_mean(panel.assign(level=pd.Categorical(panel[c].astype(str), order[c], ordered=True)),
                                        "level", "future_change"), "level",
                             c.replace("_", " ").capitalize(), f"Mean balance change over the next {HORIZON} months ($)", overall, f"all: ${overall:.0f}").properties(width=200)
              for c in profile])

alt.HConcatChart(...)

In [31]:
feature_1_insights = """
We test the customer's profile against the balance change over the next 3 months, across 1735290
examples (symmetric with heavy tails; plots above).

Income is the only steady pattern: lower-income customers grow their balances far less, from about
$100 a quarter in the Low band to $270 in Middle, $630 in UpperMiddle and $1510 in Affluent
(chart). Its effect is small next to individual movements (Kruskal-Wallis epsilon squared =
0.0025, Holm p < 0.001), and credit score and hardship show nothing (epsilon squared 0.000001 and
0.0000, both p = 0.39).
"""

In [32]:
# Do not modify this code
print_tile(size="h3", key='feature_1_insights', value=feature_1_insights)

### B.3 Explore Feature of Interest `has_loan`, `loan_risk_grade`, `debt_service_band`

In [33]:
# A loan counts at a cut-off only if it started on or before that month-end (no future loans).
panel["balance_date"] = pd.to_datetime(panel["balance_date"])
first_loan = (loans.sort_values("loan_start_date").groupby("account_id")
              .first()[["loan_start_date", "loan_risk_grade", "debt_service_band"]].reset_index())
panel = panel.merge(first_loan, on="account_id", how="left")
active = panel["loan_start_date"].notna() & (panel["loan_start_date"] <= panel["balance_date"])
panel["has_loan"] = np.where(active, "yes", "no")
panel.loc[~active, ["loan_risk_grade", "debt_service_band"]] = "no loan"
print("Examples with a loan active at the cut-off:", int(active.sum()), f"({active.mean():.1%})")

loan_fields = ["has_loan", "loan_risk_grade", "debt_service_band"]
counts = pd.concat([panel[c].value_counts().rename_axis("level").reset_index(name="examples").assign(field=c)
                    for c in loan_fields])
alt.Chart(counts).mark_bar(color="#2a78d6").encode(
    x=alt.X("examples:Q", title="Examples"), y=alt.Y("level:N", title=None),
    row=alt.Row("field:N", title=None)).properties(width=300, height=90)

Examples with a loan active at the cut-off: 323001 (18.6%)


alt.Chart(...)

In [34]:
loan_tests = pd.DataFrame({c: kruskal_epsilon(panel, "future_change", c) for c in loan_fields}).T
loan_tests["p_holm"] = holm_adjust(loan_tests["p_value"].to_numpy())
display(loan_tests)

,H,groups,n,p_value,epsilon_squared,p_holm
has_loan,306.95,2.0,1735290.0,1.010024e-68,0.000176,2.020048e-68
loan_risk_grade,311.59,6.0,1735290.0,3.227336e-65,0.000177,3.227336e-65
debt_service_band,324.16,5.0,1735290.0,6.640357e-69,0.000184,1.992107e-68


In [35]:
alt.hconcat(*[interval_chart(group_mean(panel, c, "future_change"), c, c.replace("_", " ").capitalize(), f"Mean balance change over the next {HORIZON} months ($)",
                             overall, f"all: ${overall:.0f}").properties(width=200)
              for c in loan_fields])

alt.HConcatChart(...)

In [36]:
feature_2_insights = """
18.6% of examples (323001) have a loan active at the cut-off.

Customers with an active loan grow their balances less, about $480 a quarter against $750 without
one, but the loan's risk grade and debt service band add no further gradient (chart). All three
effects are tiny (Kruskal-Wallis epsilon squared below 0.0002; the p < 0.001 only reflects the
1735290 examples).
"""

In [37]:
# Do not modify this code
print_tile(size="h3", key='feature_2_insights', value=feature_2_insights)

### B.4 Explore Feature of Interest `late_or_missed_3m` (repayment stress before the cut-off)

In [38]:
# Late or missed repayments in the LOOKBACK months up to each cut-off, for examples with an active loan
repayments["month"] = repayments["payment_date"].dt.to_period("M")
monthly_stress = (repayments.assign(stress=repayments["late_payment"] + repayments["missed_payment"])
                  .groupby(["account_id", "month"])["stress"].sum().reset_index())
with_loan = panel[panel["has_loan"] == "yes"].sort_values(["account_id", "balance_date"]).copy()
with_loan["month"] = with_loan["balance_date"].dt.to_period("M")
with_loan = with_loan.merge(monthly_stress, on=["account_id", "month"], how="left").fillna({"stress": 0})
with_loan["late_or_missed_3m"] = (with_loan.groupby("account_id")["stress"]
                                  .transform(lambda s: s.rolling(LOOKBACK, min_periods=1).sum()))
distribution_charts(with_loan["late_or_missed_3m"], f"Late or missed repayments in the {LOOKBACK} months before the cut-off", 6, 0)

Late or missed repayments in the 3 months before the cut-off: n = 323001, skewness = 4.95, excess kurtosis = 28.96, histogram omits 0 values outside the 0.0% to 100.0% quantiles


alt.HConcatChart(...)

In [39]:
display(with_loan["late_or_missed_3m"].value_counts().sort_index().to_frame("examples"))
stress_test = spearman_summary(with_loan["late_or_missed_3m"], with_loan["future_change"], "late or missed repayments")
display(stress_test.to_frame("Spearman, repayment stress vs balance change over the next HORIZON months"))

,examples
late_or_missed_3m,
0.0,297665
1.0,13987
2.0,7253
3.0,2764
4.0,1009
5.0,254
6.0,56
7.0,8
8.0,4


,"Spearman, repayment stress vs balance change over the next HORIZON months"
feature,late or missed repayments
n,323001
rho,0.002
ci_low,-0.0015
ci_high,0.0054
p_value,0.26173
magnitude,negligible


In [40]:
loan_overall = with_loan["future_change"].mean()
interval_chart(group_mean(with_loan.assign(late=with_loan["late_or_missed_3m"].clip(upper=3).astype(int)), "late", "future_change"),
               "late", "Late or missed repayments in the 3 months before (3 = three or more)", f"Mean balance change over the next {HORIZON} months ($)",
               loan_overall, f"all with a loan: ${loan_overall:.0f}")

alt.LayerChart(...)

In [41]:
feature_n_insights = """
Among the 323001 examples with an active loan, 297665 had no late or missed repayment in the
previous 3 months and 25336 had one or more (right-skewed; plots above).

Recent repayment trouble does not show in the next balance change (Spearman rho = 0.002, 95%
interval -0.0015 to 0.0054): every group stays around the $478 average for accounts with a loan
(chart).

Overall, income is the only complementary field that relates to how balances grow, and nothing
in these tables signals a coming drop.
"""

In [42]:
# Do not modify this code
print_tile(size="h3", key='feature_n_insights', value=feature_n_insights)

## C. Ethical Consideration of Features


In [43]:
# Protected attributes, recovered only to see who sits in each financial-profile group
display(panel.drop_duplicates("customer_id").groupby("credit_score_band").agg(
    customers=("customer_id", "size"), mean_age=("age", "mean"),
    vulnerable_share=("vulnerable_customer_flag", "mean"), hardship_share=("hardship_program", "mean")).round(3))

,customers,mean_age,vulnerable_share,hardship_share
credit_score_band,,,,
Excellent,1453,50.855,0.034,0.054
Fair,2929,50.831,0.036,0.048
Good,6087,50.923,0.033,0.056
Poor,1534,51.842,0.035,0.039
Very Good,2997,50.792,0.032,0.043


In [44]:
ethical_considerations = """
Credit score groups look alike in age (about 51) and in their share of vulnerable customers (3.2%
to 3.6%), with 3.9% to 5.6% in a hardship program. Income band is not a protected attribute, but
it follows socioeconomic position: a forecast that uses it expects less growth from lower-income
customers. We therefore audit any such use by customer group, and it may lead only to support,
never to a restriction.
"""

In [45]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='ethical_considerations', value=ethical_considerations)